# 04 — Self-RAG

*Level 4 — Adaptive RAG*

## Objective
Have the model critique its **own answer's** grounding after generation — a different failure mode than CRAG catches: CRAG grades evidence *before* generation; an LLM can still drift away from good evidence *during* generation, especially a smaller local model.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
for sub in ["", "self-rag"]:
    sys.path.insert(0, str(LEVEL_DIR / sub) if sub else str(LEVEL_DIR))


In [2]:
from adaptive_common.dataset import prepare
from adaptive_common.retrieval import DenseRetriever
from self_rag import self_rag_answer

data = prepare()
retriever = DenseRetriever.from_corpus(data.corpus)

bridge_item = next(q for q in data.questions.values() if q["type"] == "bridge")
print("Question:", bridge_item["question"])
print("Real answer:", bridge_item["answer"])


Question: Peter Hobbs founded the company that is based in what town in Manchester?
Real answer: Failsworth


In [3]:
result = self_rag_answer(bridge_item["question"], retriever, data.corpus, top_k=5, max_retries=1)
print("Final answer:", result["final_answer"])
print("Grounded:", result["grounded"], "| attempts:", len(result["attempts"]))
for a in result["attempts"]:
    print(f"  attempt {a['attempt']}: query={a['query_used'][:60]!r} grounded={a['grounded']}")


Final answer: The answer cannot be determined from the provided context. The question mentions Peter Wallace Hobbs, but it does not mention him founding a company based in Manchester. However, Russell Hobbs, the electrical appliance company, was founded by Bill Russell and Peter Wallace Hobbs, but the context does not specify that Peter Hobbs is the one who founded the company based in Failsworth, Greater Manchester.
Grounded: False | attempts: 2
  attempt 0: query='Peter Hobbs founded the company that is based in what town i' grounded=False
  attempt 1: query='What town in Manchester is the base of the company founded b' grounded=False


## What I observed

The final answer above is factually correct (matches the real HotpotQA answer) — but on the first run of this exact question during development, **the critique step marked a correct, genuinely-grounded answer as "ungrounded" and triggered an unnecessary retry.** That's a known, real limitation of Self-RAG with small local models: the critique model is not a perfectly reliable judge of its own (or another) generation's grounding, so a Self-RAG loop can waste a retry — or worse, in a stricter setup, discard a correct answer — based on a flawed self-critique. Self-RAG reduces hallucination risk on average; it does not eliminate it, and its critique step has its own, separate error rate worth measuring rather than assuming is zero.

## Next

[05 — Multi-Hop RAG](./05_multi_hop_rag.ipynb)
